In [3]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import requests

In [4]:
load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [5]:
MODEL='gemini-2.5-flash-lite'
Gemini_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=Gemini_BASE_URL, api_key=google_api_key)


In [6]:
links = fetch_website_links("https://nytimes.com")
links

['#site-content',
 '#site-index',
 '#after-dfp-ad-top',
 '/',
 '/',
 '/international/',
 '/ca/',
 'https://www.nytimes.com/es/',
 'https://cn.nytimes.com',
 'https://www.nytimes.com/section/todayspaper',
 '/',
 'https://www.nytimes.com/section/us',
 'https://www.nytimes.com/section/us',
 'https://www.nytimes.com/section/politics',
 'https://www.nytimes.com/section/nyregion',
 'https://www.nytimes.com/spotlight/california-news',
 'https://www.nytimes.com/section/education',
 'https://www.nytimes.com/section/health',
 'https://www.nytimes.com/section/obituaries',
 'https://www.nytimes.com/section/science',
 'https://www.nytimes.com/section/climate',
 'https://www.nytimes.com/section/weather',
 'https://www.nytimes.com/section/sports',
 'https://www.nytimes.com/section/business',
 'https://www.nytimes.com/section/technology',
 'https://www.nytimes.com/section/upshot',
 'https://www.nytimes.com/section/magazine',
 'https://www.nytimes.com/spotlight/donald-trump',
 'https://www.nytimes.com/

In [7]:
#Use lamma3.1 to read the link son the webpage and exrtract usefull links in json format

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

#user_prompt += "text"
#user_prompt = user_prompt + "text"

In [9]:
print(get_links_user_prompt("https://nytimes.com"))


Here is the list of links on the website https://nytimes.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#site-content
#site-index
#after-dfp-ad-top
/
/
/international/
/ca/
https://www.nytimes.com/es/
https://cn.nytimes.com
https://www.nytimes.com/section/todayspaper
/
https://www.nytimes.com/section/us
https://www.nytimes.com/section/us
https://www.nytimes.com/section/politics
https://www.nytimes.com/section/nyregion
https://www.nytimes.com/spotlight/california-news
https://www.nytimes.com/section/education
https://www.nytimes.com/section/health
https://www.nytimes.com/section/obituaries
https://www.nytimes.com/section/science
https://www.nytimes.com/section/climate
https://www.nytimes.com/section/weather
https://www.nytimes.com/section/sports
https://www.nytimes.com/section/business
https://www.ny

In [10]:
def select_relevant_links(url):
    response =  gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)          # strings to dict
    return links
    

In [11]:
select_relevant_links("https://nytimes.com")

{'links': [{'type': 'about page', 'url': 'https://www.nytco.com/'},
  {'type': 'careers page', 'url': 'https://www.nytco.com/careers/'},
  {'type': 'company information', 'url': 'https://advertising.nytimes.com/'}]}

In [12]:
select_relevant_links("https://cricbuzz.com")

{'links': [{'type': 'company information',
   'url': 'https://cricbuzz.com/info/about'},
  {'type': 'careers page', 'url': 'https://cricbuzz.com/careers'},
  {'type': 'product information',
   'url': 'https://cricbuzz.com/product-blog/cricbuzz-mobile-apps-tv-ad-cricket-ka-keeda'},
  {'type': 'advertising information',
   'url': 'https://cricbuzz.com/info/advertise'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://nytimes.com")

Selecting relevant links for https://nytimes.com by calling gemini-2.5-flash-lite
Found 4 relevant links


{'links': [{'type': 'company information', 'url': 'https://www.nytco.com/'},
  {'type': 'careers page', 'url': 'https://www.nytco.com/careers/'},
  {'type': 'about page',
   'url': 'https://help.nytimes.com/hc/en-us/articles/115015727108-Accessibility'},
  {'type': 'about page',
   'url': 'https://help.nytimes.com/hc/en-us/articles/115015385887-Contact-The-New-York-Times'}]}

In [15]:
select_relevant_links("https://cricbuzz.com")

Selecting relevant links for https://cricbuzz.com by calling gemini-2.5-flash-lite
Found 7 relevant links


{'links': [{'type': 'about page',
   'url': 'https://www.cricbuzz.com/info/about'},
  {'type': 'careers page', 'url': 'https://www.cricbuzz.com/careers'},
  {'type': 'product page',
   'url': 'https://www.cricbuzz.com/product-blog/cricbuzz-mobile-apps-tv-ad-cricket-ka-keeda'},
  {'type': 'social media', 'url': 'https://www.facebook.com/cricbuzz'},
  {'type': 'social media', 'url': 'https://twitter.com/cricbuzz'},
  {'type': 'social media', 'url': 'https://www.youtube.com/c/cricbuzz'},
  {'type': 'social media', 'url': 'https://in.pinterest.com/cricbuzz/'}]}

In [16]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 14 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'models', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'brand', 'url': 'https://huggingface.co/brand'},
  {'type': 'company social media', 'url': 'https://github.com/huggingface'},
  {'type': 'company social media', 'url': 'https://twitter.com/huggingface'},
  {'type': 'company social media',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [17]:
#Assemble all the details into another prompt to gemini
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://nytimes.com"))

Selecting relevant links for https://nytimes.com by calling gemini-2.5-flash-lite
Found 3 relevant links
## Landing Page:

The New York Times - Breaking News, US News, World News and Videos

Skip to content
Skip to site index
SKIP ADVERTISEMENT
U.S.
International
Canada
Español
中文
Today’s Paper
U.S.
Sections
U.S.
Politics
New York
California
Education
Health
Obituaries
Science
Climate
Weather
Sports
Business
Tech
The Upshot
The Magazine
Top Stories
Donald Trump
Supreme Court
Congress
Immigration
Abortion
Newsletters
The Morning
Make sense of the day’s news and ideas.
The Evening
Catch up on big news, and wind down to end your day.
See all newsletters
Podcasts
The Daily
The biggest stories of our time, in 20 minutes a day.
See all podcasts
World
Sections
World
Africa
Americas
Asia
Australia
Canada
Europe
Middle East
Science
Climate
Weather
Health
Obituaries
Top Stories
Middle East Crisis
Russia-Ukraine War
China International Relations
The Global Profile
Leer en Español
Newsletters
The 

In [31]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customersand careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 14 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-Image\nUpdated\n4 days ago\n•\n7.59k\n•\n843\nLightricks/LTX-2\nUpdated\nabout 17 hours ago\n•\n1.54M\n•\n1.16k\nfal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA\nUpdated\n12 days ago\n•\n55.2k\n•\n756\nopenbmb/AgentCPM-Explore\nUpdated\nabout 20 hours ago\n•\n1.83k\n•\n350\ngoogle/translategemma-4b-it\nUpdated\n4 days ago\n•\n17.6k\n•\n321\nBrowse 2M+ models\nSpaces\nRunning\non\n

In [24]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 12 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the leading collaboration platform for the machine learning community. We empower developers, scientists, and enthusiasts to create, discover, and share open-source AI models, datasets, and applications. Join a vibrant community and be at the heart of the AI revolution.

## What We Offer

*   **The Hugging Face Hub:** A central platform to host, explore, discover, and experiment with over 2 million open-source ML models, 500k+ datasets, and 1 million+ applications across various modalities including text, image, video, audio, and 3D.
*   **Open Source ML Stack:** Move faster with our robust and widely used open-source libraries and tools.
*   **AI Apps:** Explore and deploy cutting-edge AI applications built by the community.
*   **Collaboration:** A space to share your work, build your ML profile, and collaborate with others.

## For Our Community

Hugging Face is more than just a platform; it's a growing community dedicated to building an open and ethical AI future. We believe in the power of open source to accelerate innovation and democratize access to AI.

## Pricing & Enterprise Solutions

We offer flexible pricing to suit individual needs, growing teams, and large enterprises.

*   **PRO:** Enhance your personal experience with increased storage, inference credits, and ZeroGPU quota. ($9/month)
*   **Team:** Ideal for growing teams with features like SSO, SAML support, granular access control, and advanced compute options. ($20/user/month)
*   **Enterprise:** Comprehensive solutions with custom onboarding, highest limits, managed billing, personalized support, and legal/compliance assistance. (Starting at $50/user/month)

We also offer Expert Support for organizations looking to adopt the Hugging Face Hub.

## Careers

Hugging Face is at the forefront of the AI revolution, driven by a talented science team and a fast-growing community. We are looking for passionate individuals to join us in building the future of AI. Explore our current openings on our [Careers Page](https://huggingface.co/join).

## Connect With Us

*   **Website:** [huggingface.co](https://huggingface.co)
*   **Community:** Explore our platform, engage in discussions, and share your work.
*   **Social Media:** Follow us on GitHub, Twitter, and LinkedIn.
*   **Resources:** Access our extensive Documentation, Blog, and Learn sections.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [29]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True  # streanms one by one in chunks (parts)
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 13 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the premier platform where the machine learning community collaborates to build the future of AI. We provide a central hub for discovering, sharing, and deploying state-of-the-art AI models, datasets, and applications.

## What We Offer:

*   **2M+ Models:** Explore a vast and growing collection of machine learning models for various tasks, including text generation, image-to-text, text-to-image, and much more.
*   **1M+ Applications (Spaces):** Discover and interact with a wide range of AI applications built by the community, from image manipulation tools to advanced text generators.
*   **500k+ Datasets:** Access a diverse range of datasets to train and fine-tune your own AI models.

## For Our Community:

Hugging Face is more than just a platform; it's a vibrant community. We empower individuals and teams to:

*   **Collaborate:** Host and work together on unlimited public models, datasets, and applications.
*   **Innovate:** Leverage the HF Open Source stack to accelerate your development.
*   **Showcase:** Build your ML portfolio and share your work with the world.
*   **Explore Modalities:** Engage with AI across text, image, video, audio, and even 3D.

## For Enterprise:

We offer enterprise solutions to help your organization harness the power of AI. Accelerate your ML initiatives with our paid compute and dedicated support.

## Company Culture:

Hugging Face is dedicated to fostering an open and collaborative environment. We believe in the power of community-driven innovation and are committed to making AI accessible to everyone. Our culture encourages learning, sharing, and pushing the boundaries of what's possible in machine learning.

## Careers:

Join a team at the forefront of AI innovation. We are always looking for passionate individuals to help us build the future of machine learning. Explore opportunities to contribute to a dynamic and impactful company.

## Contact Us:

For inquiries, please visit our website and navigate to the "Contact Us" section or explore our documentation for support.

In [32]:
#Humorous system prompt
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Found 6 relevant links


## Hugging Face: Where AI Gets Hugs (and Builds the Future!)

Welcome to Hugging Face, the place where AI isn't just built, it's *hugged* into existence! Think of us as the ultimate playground for AI enthusiasts, a bustling metropolis of code, data, and dazzling applications.

### What's All the Hugging About?

We're the AI community that's building the future, one model, dataset, and space at a time. Whether you're a seasoned ML guru or just dipping your toes into the AI ocean, Hugging Face is your launchpad. Explore over **2 million models** (that's a lot of digital brains!), **over 500,000 datasets** (the fuel for those brains), and **over 1 million applications** (where the magic happens!).

### For the Dreamers and Doers:

*   **Models:** Got a brilliant idea for an AI? Train it, share it, and let the world marvel. Or, borrow and tweak from the millions already here. It's like a digital library, but with more "wow."
*   **Datasets:** From tiny text snippets to colossal image collections, we've got the data to make your AI dreams a reality. Just don't get lost in the data jungle!
*   **Spaces:** This is where the fun truly lives! Deploy and showcase your AI apps. Think of it as your personal AI art gallery, where your creations run live for everyone to play with. Warning: may cause spontaneous applause and requests for selfies with your algorithms.

### Our Vibe: Community, Collaboration, and Caffeine (Probably)

We're all about building AI *together*. Our culture is a vibrant mix of open-source spirit, relentless innovation, and a shared passion for making AI accessible. We believe in moving fast, breaking things (and then fixing them with even better code), and celebrating every breakthrough.

### Got the AI Bug? Join Our Flock!

We're always looking for brilliant minds to join our quest. If you're passionate about AI, love a good collaborative challenge, and don't mind the occasional enthusiastic "Hugging Face!" greeting, you might just find your tribe here. Check out our **careers page** for openings – we promise it's more exciting than sorting socks.

### For the Big Leagues: Enterprise Hugs!

For organizations ready to scale their AI game, we offer enterprise-grade solutions. Think advanced security, dedicated support, and enough compute power to make your AI sing. We've got plans for teams and bespoke solutions for enterprises – because even the biggest AI ambitions deserve a personalized hug.

### Pricing: We've Got a Hug for Every Budget

Whether you're a solo coder needing a little extra oomph with our **PRO Account** (for just $9/month, you get boosted storage, credits, and priority queue access – practically a steal!) or a team looking to collaborate, we have flexible options. Check out our **Pricing page** for details.

**So, come on over! Let's build the future of AI, together. And maybe share some snacks. Hugging Face: It's more than a company, it's a movement.**

In [35]:
stream_brochure("CricBuzz", "https://nationalgeographic.com")

Selecting relevant links for https://nationalgeographic.com by calling gemini-2.5-flash-lite
Found 13 relevant links


### National Geographic: Go Deeper, Laugh Louder!

**Tired of the mundane? Do your eyeballs yearn for more than just the latest cat video? Welcome to National Geographic, where we explore the world, one mind-blowing fact at a time!**

**What We Do (Besides Making You Say "Wow!")**

We're not just a magazine, people! We're your passport to the planet's most fascinating corners, from the deepest oceans to the loftiest ambitions of scientists trying to defeat *moon dust* (seriously, who knew dust was so devious?). We bring you stories that are so incredible, you’ll question if reality is just a really well-produced documentary.

*   **Science that makes your brain do backflips:** Is it raining or snowing? We've got the deets. Gestational diabetes got you down? We're on it. Why are women more migraine-prone? Apparently, it's complicated. We dive deep so you don't have to (unless you want to, then by all means, dive deep!).
*   **History that’s cooler than your grandpa's stories:** Ancient Romans? World’s greatest builders. 5,000-year-old harpoons? Apparently, whaling is *old*. We unearth the past so you can brag about it at your next trivia night.
*   **Travel that sparks your wanderlust (and maybe a little envy):** From charming small towns to the uncertain future of the world's most expensive spice (yes, saffron is *that* dramatic), we’ll make you want to pack your bags.

**Our Customers: The Curious, The Courageous, The "Are You Kidding Me?!" Bunch**

If you’re the kind of person who asks "why?" more often than "what's for dinner?", you're one of us. Our audience is a magnificent mix of intrepid explorers, armchair adventurers, and anyone who believes the world is an endlessly fascinating place. You're the ones who subscribe for full access because, let's be honest, you can't get enough.

**National Geographic Live: Because Watching on TV is So Last Century!**

Ever wanted to meet the folks who are actually *doing* the exploring? Now you can! We bring our incredible National Geographic Explorers to a stage near you. Expect behind-the-scenes stories, stunning visuals, and enough inspiration to make you want to book your own expedition (we have packages, just saying). Plus, pre-show trivia – because who doesn't love bragging about knowing things?

**Careers: Join the Expedition!**

Think you've got what it takes to explore, discover, and tell tales that will blow minds? We're looking for passionate individuals to join our team. Whether you're a scientist, a storyteller, an organizer of epic events, or someone who can actually defeat moon dust, we might have a place for you. Check out our careers page – adventure awaits! (And maybe a really cool office with a globe and a telescope.)

**National Geographic: We're Not Just Exploring the World, We're Making it More Interesting. Join Us!**

In [36]:
stream_brochure("VU pune", "https://vupune.ac.in")

Selecting relevant links for https://vupune.ac.in by calling gemini-2.5-flash-lite


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 48.678399793s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash